# 학생 문자 시스템 2-1 — 정상평가 피드백 생성기

제1·2·4·5차 정기평가의 반별 `*-정제.xlsx`를 읽어 학생별 학부모 문자 전체를 만들고, 반별 TXT로 저장합니다.

- 제3·6차 누적평가는 이 노트북에서 의도적으로 중단합니다.
- 점수·영역 결과는 Excel의 값만 사용합니다.
- 수업 태도·과제 관련 내용은 점수로 추정하지 않고 `FeedbackInput.xlsx`에 실제 관찰 문구가 있을 때만 넣습니다.
- 결과 TXT는 UTF-8 BOM으로 저장하고, 기존 파일은 덮어쓰지 않습니다.

아래의 **1. 설정**만 확인한 뒤 `런타임 → 모두 실행`하면 됩니다.


## 1. 설정

Windows 경로가 실제 폴더와 다르면 이 셀만 고쳐 주세요.

`COMMON_LESSON_TEXT_PATH`에는 1단계에서 만든 수업내용 TXT 하나를 지정할 수 있습니다. 반마다 다른 파일을 쓰려면 `CLASS_LESSON_TEXT_PATHS`에 `학급명: 파일경로`를 적습니다. 둘 다 `None`/빈 딕셔너리면 수업내용 TXT는 삽입하지 않습니다.

마무리 문구는 규정서에서 아직 미확정이므로 기본값이 빈 문자열입니다. 확정되면 `CLOSING_BY_EXAM`에 입력하면 됩니다.


In [ ]:
from pathlib import Path

DATA_CLEAN_PATH = Path(r"C:\Users\9191h\Desktop\TestFeedback\DataClean")
FEEDBACK_INPUT_PATH = Path(r"C:\Users\9191h\Desktop\TestFeedback\FeedbackInput.xlsx")
FEEDBACK_OUTPUT_PATH = Path(r"C:\Users\9191h\Desktop\TestFeedback\Feedback_Result")

# 선택: 모든 반에 공통으로 넣을 수업내용 TXT
COMMON_LESSON_TEXT_PATH = None

# 선택: 반별 수업내용 TXT. 예: {"선랑-토0930-임서영T": Path(r"C:\...\중1-[8호].txt")}
CLASS_LESSON_TEXT_PATHS = {}

# 규정서에서 아직 미확정인 마무리 문구. 확정된 문장만 넣으세요.
CLOSING_BY_EXAM = {
    1: "",
    2: "",
    4: "",
    5: "",
}

INCLUDE_CLASS_AVERAGE = True
INCLUDE_CLASS_FIRST_PLACE = True
STRICT_FILENAME_CLASS_MATCH = True
MAX_RECOMMENDED_MESSAGE_CHARS = 1000

# 수동문구의 맞춤법을 일괄 치환하려면 정확한 치환만 등록합니다.
# 예: {"됬습니다": "됐습니다"}
MANUAL_TEXT_REPLACEMENTS = {}


## 2. 규칙과 공통 함수

확정 규칙은 코드에 고정했습니다. 규정서에서 문구가 비어 있는 `구조 파악`·`추론`은 근거 없는 해석 대신 점수·득점률만 알리는 중립 문구를 사용합니다.


In [ ]:
import re
from dataclasses import dataclass
from datetime import datetime
from decimal import Decimal, InvalidOperation, ROUND_HALF_UP
from typing import Any

from openpyxl import load_workbook


ALLOWED_EXAMS = {1, 2, 4, 5}
AREA_ORDER = ["어휘 파악", "핵심 요지 파악", "내용 파악", "문맥 파악", "구조 파악", "추론"]
AREA_MAX = {
    "어휘 파악": Decimal("17"),
    "핵심 요지 파악": Decimal("7"),
    "내용 파악": Decimal("21"),
    "문맥 파악": Decimal("12"),
    "구조 파악": Decimal("12"),
    "추론": Decimal("31"),
}
EXPECTED_COLUMNS = {
    1: ["학급명", "학생명", "학번", "점수", *AREA_ORDER],
    2: ["학급명", "학생명", "학번", "점수", *AREA_ORDER, "이전점수"],
    4: ["학급명", "학생명", "학번", "점수", *AREA_ORDER, "이전점수"],
    5: ["학급명", "학생명", "학번", "점수", *AREA_ORDER, "이전점수"],
}
EXCLUSION_CODES = {
    "CURRENT_SCORE", "SCORE_CHANGE", "OVERALL_EVALUATION",
    "AREA_STRENGTH", "AREA_WEAKNESS", "AREA_GENERAL", "CLASS_AVERAGE",
    "CLASS_RANK", "LESSON_CONTENT", "STUDENT_NOTE", "CLASS_NOTE",
}


class FeedbackError(RuntimeError):
    """입력값이나 규칙 위반으로 최종 TXT 생성을 중단해야 할 때 사용합니다."""


@dataclass
class ClassData:
    path: Path
    year: int
    exam_no: int
    exam_name: str
    class_name: str
    rows: list[dict[str, Any]]


@dataclass
class FeedbackInput:
    student_blocks: dict[str, list[str]]
    student_lessons: dict[str, list[str]]
    student_exclusions: dict[str, set[str]]
    class_blocks: dict[str, list[str]]
    class_lessons: dict[str, list[str]]
    supplied_class_averages: dict[str, Decimal]
    warnings: list[str]


def clean_text(value: Any) -> str:
    if value is None:
        return ""
    return str(value).replace("\u00a0", " ").strip()


def is_missing(value: Any) -> bool:
    return clean_text(value) == ""


def context_text(path: Path, row: dict[str, Any], column: str, value: Any) -> str:
    return (
        f"파일명={path.name}, Excel행={row.get('_excel_row', '?')}, "
        f"학급명={clean_text(row.get('학급명')) or '?'}, "
        f"학생명={clean_text(row.get('학생명')) or '?'}, "
        f"학번={clean_text(row.get('학번')) or '?'}, "
        f"열={column}, 실제값={value!r}"
    )


def to_decimal(value: Any, *, path: Path, row: dict[str, Any], column: str) -> Decimal:
    raw = clean_text(value).replace(",", "")
    if not raw:
        raise FeedbackError("숫자 결측: " + context_text(path, row, column, value))
    try:
        number = Decimal(raw)
    except (InvalidOperation, ValueError):
        raise FeedbackError("숫자로 변환할 수 없음: " + context_text(path, row, column, value))
    if not number.is_finite():
        raise FeedbackError("유한한 숫자가 아님: " + context_text(path, row, column, value))
    return number


def format_number(value: Decimal) -> str:
    if value == value.to_integral_value():
        return str(int(value))
    return format(value.normalize(), "f")


def format_derived(value: Decimal, places: str = "0.01") -> str:
    rounded = value.quantize(Decimal(places), rounding=ROUND_HALF_UP)
    return format_number(rounded)


def has_final_consonant(text: str) -> bool:
    if not text:
        raise FeedbackError("학생명의 이름 부분이 비어 있습니다.")
    code = ord(text[-1])
    if not (0xAC00 <= code <= 0xD7A3):
        raise FeedbackError(f"한글로 끝나지 않는 학생명은 처리할 수 없습니다: {text!r}")
    return (code - 0xAC00) % 28 != 0


def given_name(full_name: str) -> str:
    name = re.sub(r"\s+", "", clean_text(full_name))
    if len(name) < 2:
        raise FeedbackError(f"성을 제외한 이름을 만들 수 없습니다: {full_name!r}")
    return name[1:]


def friendly_subject(full_name: str) -> str:
    name = given_name(full_name)
    return name + ("이는" if has_final_consonant(name) else "는")


def friendly_genitive(full_name: str) -> str:
    name = given_name(full_name)
    return name + ("이의" if has_final_consonant(name) else "의")


def normalize_manual_text(value: Any) -> str:
    text = clean_text(value)
    for old, new in MANUAL_TEXT_REPLACEMENTS.items():
        text = text.replace(old, new)
    if text and text[-1] not in ".!?。！？★":
        text += "."
    return text


def parse_exam_metadata(a1: Any, path: Path) -> tuple[int, int, str]:
    text = re.sub(r"\s+", " ", clean_text(a1))
    year_match = re.search(r"((?:19|20)\d{2})\s*년?", text)
    exam_match = re.search(r"제\s*(\d+)\s*차", text)
    if not year_match or not exam_match:
        raise FeedbackError(f"A1에서 시험 연도·차수를 찾을 수 없습니다: 파일명={path.name}, A1={a1!r}")
    year = int(year_match.group(1))
    exam_no = int(exam_match.group(1))
    if exam_no not in ALLOWED_EXAMS:
        raise FeedbackError(
            f"이 노트북은 제1·2·4·5차만 처리합니다: 파일명={path.name}, 감지 차수={exam_no}차"
        )
    return year, exam_no, f"{year}년 제{exam_no}차 정기평가"


def student_id_from_cell(cell) -> str:
    value = cell.value
    if value is None:
        return ""
    if isinstance(value, bool):
        return str(value)
    if isinstance(value, int):
        fmt = clean_text(cell.number_format)
        if fmt and re.fullmatch(r"0+", fmt):
            return f"{value:0{len(fmt)}d}"
        return str(value)
    if isinstance(value, float) and value.is_integer():
        fmt = clean_text(cell.number_format)
        if fmt and re.fullmatch(r"0+", fmt):
            return f"{int(value):0{len(fmt)}d}"
        return str(int(value))
    return clean_text(value)


## 3. Excel 입력 검증

A1의 시험 연도·차수, 필수 열, 학급명, 학번 중복, 점수·영역 합계와 배점을 검사합니다. 한 건이라도 오류가 있으면 TXT를 쓰기 전에 전체 실행을 중단합니다.


In [ ]:
def find_header_row(ws, required_markers: set[str], max_rows: int = 20) -> int:
    for row_no in range(1, min(ws.max_row, max_rows) + 1):
        values = {clean_text(ws.cell(row_no, col).value) for col in range(1, ws.max_column + 1)}
        if required_markers.issubset(values):
            return row_no
    raise FeedbackError(
        f"시트 '{ws.title}'의 앞 {max_rows}행에서 헤더 {sorted(required_markers)}를 찾지 못했습니다."
    )


def load_class_file(path: Path) -> tuple[ClassData, list[str]]:
    warnings = []
    wb = load_workbook(path, data_only=True, read_only=False)
    ws = wb.active
    year, exam_no, exam_name = parse_exam_metadata(ws["A1"].value, path)

    header_row = find_header_row(ws, {"학급명", "학생명", "학번", "점수"})
    headers = [clean_text(ws.cell(header_row, col).value) for col in range(1, ws.max_column + 1)]
    while headers and headers[-1] == "":
        headers.pop()
    if len(headers) != len(set(headers)):
        duplicates = sorted({h for h in headers if h and headers.count(h) > 1})
        raise FeedbackError(f"중복된 열 이름: 파일명={path.name}, 열={duplicates}")

    expected = EXPECTED_COLUMNS[exam_no]
    if headers != expected:
        raise FeedbackError(
            f"필수 열 또는 열 순서가 다릅니다: 파일명={path.name}\n"
            f"기대={expected}\n실제={headers}"
        )

    id_col = headers.index("학번") + 1
    rows = []
    for excel_row in range(header_row + 1, ws.max_row + 1):
        values = [ws.cell(excel_row, col).value for col in range(1, len(headers) + 1)]
        if all(is_missing(v) for v in values):
            continue
        record = dict(zip(headers, values))
        record["학번"] = student_id_from_cell(ws.cell(excel_row, id_col))
        record["_excel_row"] = excel_row
        rows.append(record)

    if not rows:
        raise FeedbackError(f"학생 자료가 없습니다: 파일명={path.name}")

    for row in rows:
        for col in ("학급명", "학생명", "학번"):
            if is_missing(row.get(col)):
                raise FeedbackError("필수값 결측: " + context_text(path, row, col, row.get(col)))
        row["학급명"] = clean_text(row["학급명"])
        row["학생명"] = clean_text(row["학생명"])
        row["학번"] = clean_text(row["학번"])

    class_names = {row["학급명"] for row in rows}
    if len(class_names) != 1:
        raise FeedbackError(f"한 파일에 학급명이 여러 개입니다: 파일명={path.name}, 학급명={sorted(class_names)}")
    class_name = next(iter(class_names))

    ids = [row["학번"] for row in rows]
    duplicates = sorted({sid for sid in ids if ids.count(sid) > 1})
    if duplicates:
        raise FeedbackError(f"같은 파일에 학번이 중복되었습니다: 파일명={path.name}, 학번={duplicates}")

    if class_name not in path.stem:
        message = f"파일명에서 학급명을 확인하지 못했습니다: 파일명={path.name}, 학급명={class_name}"
        if STRICT_FILENAME_CLASS_MATCH:
            raise FeedbackError(message)
        warnings.append(message)

    return ClassData(path, year, exam_no, exam_name, class_name, rows), warnings


def analyze_row(data: ClassData, row: dict[str, Any]) -> dict[str, Any]:
    path = data.path
    current_missing = is_missing(row.get("점수"))
    if current_missing:
        return {
            "absent": True,
            "current": None,
            "previous": None,
            "areas": [],
        }

    current = to_decimal(row["점수"], path=path, row=row, column="점수")
    if current < 0 or current > 100:
        raise FeedbackError("현재점수가 0~100 범위를 벗어남: " + context_text(path, row, "점수", row["점수"]))

    previous = None
    if data.exam_no in {2, 4, 5} and not is_missing(row.get("이전점수")):
        previous = to_decimal(row["이전점수"], path=path, row=row, column="이전점수")
        if previous < 0 or previous > 100:
            raise FeedbackError(
                "이전점수가 0~100 범위를 벗어남: " + context_text(path, row, "이전점수", row["이전점수"])
            )

    areas = []
    for index, area in enumerate(AREA_ORDER):
        if is_missing(row.get(area)):
            raise FeedbackError("일부 영역 점수 결측: " + context_text(path, row, area, row.get(area)))
        score = to_decimal(row[area], path=path, row=row, column=area)
        maximum = AREA_MAX[area]
        if score < 0 or score > maximum:
            raise FeedbackError(
                f"영역 점수가 배점 범위를 벗어남(기대 0~{format_number(maximum)}): "
                + context_text(path, row, area, row[area])
            )
        rate = score / maximum * Decimal("100")
        if score == 0:
            state = "zero"
        elif score == maximum:
            state = "perfect"
        elif rate <= 50:
            state = "weak"
        elif rate >= 80:
            state = "strong"
        else:
            state = "general"
        areas.append({
            "name": area,
            "index": index,
            "score": score,
            "max": maximum,
            "rate": rate,
            "state": state,
        })

    total = sum((item["score"] for item in areas), Decimal("0"))
    numbers = [current, *[item["score"] for item in areas]]
    has_decimal = any(number != number.to_integral_value() for number in numbers)
    tolerance = Decimal("0.01") if has_decimal else Decimal("0")
    if abs(total - current) > tolerance:
        raise FeedbackError(
            f"점수와 여섯 영역 합계 불일치: {context_text(path, row, '점수', row['점수'])}, "
            f"영역합계={format_number(total)}, 허용오차={format_number(tolerance)}"
        )

    return {
        "absent": False,
        "current": current,
        "previous": previous,
        "areas": areas,
    }


## 4. 수동 입력과 수업내용

`FeedbackInput.xlsx`가 없으면 자동문구만 생성합니다. 파일이 있으면 다음 열을 인식합니다.

- 학생 식별: `학번` (필수), `학생명`·`시험연도`·`시험차수`·`학급명` (검증/필터용)
- 학생 문구: `반드시 포함할 문장`, `학생메모`, `교사 관찰 메모`, `과제 피드백`, `수업 태도`
- 수업내용: `수업내용`
- 자동문구 제외: `자동문구 제외 항목`에 코드를 쉼표로 구분
- 학급 문구: `학급메모`, `공지 문장`, `공통 오답 특징`, `반드시 포함할 문장`

학생·학급메모 시트를 나눠도 되고, 한 시트에 함께 적어도 됩니다. 수동문구는 실제 관찰 문장으로 간주해 내용 자체를 새로 추론하지 않습니다.


In [ ]:
STUDENT_TEXT_COLUMNS = ["반드시 포함할 문장", "학생메모", "교사 관찰 메모", "과제 피드백", "수업 태도"]
CLASS_TEXT_COLUMNS = ["학급메모", "공지 문장", "공통 오답 특징", "반드시 포함할 문장"]


def sheet_records(ws) -> list[dict[str, Any]]:
    header_row = None
    for row_no in range(1, min(ws.max_row, 20) + 1):
        values = [clean_text(ws.cell(row_no, col).value) for col in range(1, ws.max_column + 1)]
        if "학번" in values or "학급명" in values:
            header_row = row_no
            break
    if header_row is None:
        return []

    headers = [clean_text(ws.cell(header_row, col).value) for col in range(1, ws.max_column + 1)]
    while headers and headers[-1] == "":
        headers.pop()
    if len([h for h in headers if h]) != len(set(h for h in headers if h)):
        raise FeedbackError(f"FeedbackInput 시트 '{ws.title}'에 중복 열 이름이 있습니다.")

    id_col = headers.index("학번") + 1 if "학번" in headers else None
    records = []
    for excel_row in range(header_row + 1, ws.max_row + 1):
        values = [ws.cell(excel_row, col).value for col in range(1, len(headers) + 1)]
        if all(is_missing(v) for v in values):
            continue
        record = {header: value for header, value in zip(headers, values) if header}
        if id_col:
            record["학번"] = student_id_from_cell(ws.cell(excel_row, id_col))
        record["_excel_row"] = excel_row
        record["_sheet"] = ws.title
        records.append(record)
    return records


def row_matches_exam(row: dict[str, Any], year: int, exam_no: int) -> bool:
    if not is_missing(row.get("시험연도")):
        try:
            if int(Decimal(clean_text(row["시험연도"]))) != year:
                return False
        except Exception:
            raise FeedbackError(
                f"FeedbackInput 시험연도를 해석할 수 없습니다: 시트={row.get('_sheet')}, "
                f"Excel행={row.get('_excel_row')}, 값={row.get('시험연도')!r}"
            )
    if not is_missing(row.get("시험차수")):
        raw = clean_text(row["시험차수"])
        match = re.search(r"(\d+)", raw)
        if not match:
            raise FeedbackError(
                f"FeedbackInput 시험차수를 해석할 수 없습니다: 시트={row.get('_sheet')}, "
                f"Excel행={row.get('_excel_row')}, 값={row.get('시험차수')!r}"
            )
        if int(match.group(1)) != exam_no:
            return False
    return True


def split_exclusion_codes(value: Any) -> set[str]:
    if is_missing(value):
        return set()
    codes = {code.strip().upper() for code in re.split(r"[,;/\s]+", clean_text(value)) if code.strip()}
    unknown = codes - EXCLUSION_CODES
    if unknown:
        raise FeedbackError(f"정의되지 않은 자동문구 제외 코드: {sorted(unknown)}")
    return codes


def append_unique(mapping: dict[str, list[str]], key: str, value: str) -> None:
    if not value:
        return
    current = mapping.setdefault(key, [])
    if value not in current:
        current.append(value)


def load_feedback_input(
    path: Path,
    year: int,
    exam_no: int,
    students_by_id: dict[str, dict[str, Any]],
    class_names: set[str],
) -> FeedbackInput:
    result = FeedbackInput({}, {}, {}, {}, {}, {}, [])
    if not path.exists():
        result.warnings.append(f"FeedbackInput.xlsx가 없어 수동문구 없이 진행합니다: {path}")
        return result

    wb = load_workbook(path, data_only=True, read_only=False)
    seen_student_rows = set()
    seen_class_rows = set()
    class_average_values: dict[str, set[Decimal]] = {}

    for ws in wb.worksheets:
        for row in sheet_records(ws):
            if not row_matches_exam(row, year, exam_no):
                continue

            sid = clean_text(row.get("학번"))
            class_name = clean_text(row.get("학급명"))

            if sid:
                identity = (year, exam_no, sid)
                if identity in seen_student_rows:
                    raise FeedbackError(
                        f"FeedbackInput 학생 행 중복: 시트={ws.title}, Excel행={row['_excel_row']}, 학번={sid}"
                    )
                seen_student_rows.add(identity)
                if sid not in students_by_id:
                    result.warnings.append(f"현재 정제 파일에 없는 학번의 수동문구를 건너뜁니다: 학번={sid}")
                    continue
                student = students_by_id[sid]
                supplied_name = clean_text(row.get("학생명"))
                if supplied_name and supplied_name != student["학생명"]:
                    raise FeedbackError(
                        f"학번과 학생명이 일치하지 않습니다: 시트={ws.title}, Excel행={row['_excel_row']}, "
                        f"학번={sid}, 정제파일={student['학생명']}, FeedbackInput={supplied_name}"
                    )
                supplied_class = class_name
                if supplied_class and supplied_class != student["학급명"]:
                    raise FeedbackError(
                        f"학번과 학급명이 일치하지 않습니다: 시트={ws.title}, Excel행={row['_excel_row']}, "
                        f"학번={sid}, 정제파일={student['학급명']}, FeedbackInput={supplied_class}"
                    )

                for column in STUDENT_TEXT_COLUMNS:
                    if not is_missing(row.get(column)):
                        append_unique(result.student_blocks, sid, normalize_manual_text(row[column]))
                if not is_missing(row.get("수업내용")):
                    append_unique(result.student_lessons, sid, normalize_manual_text(row["수업내용"]))
                result.student_exclusions[sid] = split_exclusion_codes(row.get("자동문구 제외 항목"))

            # 학생 행의 학급명은 신원 검증용입니다. 학생 문구를 학급 전체에 퍼뜨리지 않습니다.
            if class_name and not sid:
                if class_name not in class_names:
                    result.warnings.append(f"현재 정제 파일에 없는 학급의 수동문구를 건너뜁니다: 학급명={class_name}")
                    continue
                identity = (year, exam_no, class_name)
                if identity in seen_class_rows:
                    raise FeedbackError(
                        f"FeedbackInput 학급 행 중복: 시트={ws.title}, Excel행={row['_excel_row']}, "
                        f"학급명={class_name}"
                    )
                seen_class_rows.add(identity)
                for column in CLASS_TEXT_COLUMNS:
                    if not is_missing(row.get(column)):
                        append_unique(result.class_blocks, class_name, normalize_manual_text(row[column]))
                if not is_missing(row.get("수업내용")):
                    append_unique(result.class_lessons, class_name, normalize_manual_text(row["수업내용"]))

            # 학급평균은 학생별 입력 행에 반복되어 있어도 같은 값이면 허용합니다.
            if class_name and class_name in class_names and not is_missing(row.get("학급평균")):
                value = to_decimal(row["학급평균"], path=path, row=row, column="학급평균")
                class_average_values.setdefault(class_name, set()).add(value)

    for class_name, values in class_average_values.items():
        if len(values) > 1:
            raise FeedbackError(f"FeedbackInput의 학급평균 값이 서로 다릅니다: 학급명={class_name}, 값={sorted(values)}")
        result.supplied_class_averages[class_name] = next(iter(values))

    return result


def read_text_file(path_value: Any) -> str:
    if not path_value:
        return ""
    path = Path(path_value)
    if not path.exists():
        raise FeedbackError(f"수업내용 TXT를 찾을 수 없습니다: {path}")
    for encoding in ("utf-8-sig", "utf-8", "cp949"):
        try:
            text = path.read_text(encoding=encoding).strip()
            return text
        except UnicodeDecodeError:
            continue
    raise FeedbackError(f"수업내용 TXT 인코딩을 해석할 수 없습니다: {path}")


## 5. 문구 조립

강점은 득점률 80% 이상, 취약점은 50% 이하입니다. 각각 최대 3개를 고르며, 경계에서 같은 득점률이면 동률 영역을 함께 출력합니다. 4개 이상의 취약 영역이 있으면 통합 안내 뒤 가장 낮은 3개 영역의 학습 설명을 붙입니다.


In [ ]:
AREA_MESSAGES = {
    "어휘 파악": {
        "zero": "'어휘'가 0점입니다. 문해력의 기초가 되는 부분으로 어휘가 안정되어야 더 어려운 영역도 순차적으로 향상될 수 있습니다. 과제를 할 때 어휘를 사전적 설명으로만 외우면 실제 문제에 적용되지 않는 경우가 많기 때문에, 자신이 아는 쉬운 단어로 바꾸어 익히는 연습이 도움이 될 것입니다.",
        "perfect": "'어휘'가 만점으로 문해력의 기초가 되는 어휘를 잘 익히고 있는 것으로 보입니다.",
        "weak": "'어휘'의 득점률은 {rate}%입니다. 어휘는 문해력의 기초가 되는 부분으로, 과제를 할 때 뜻을 자신이 아는 쉬운 말로 바꾸어 익히는 연습이 도움이 될 것입니다.",
        "strong": "'어휘'의 득점률은 {rate}%로 문해력의 기초가 되는 어휘를 잘 익히고 있는 것으로 보입니다.",
        "general": "'어휘'에서 실점이 확인되었습니다.",
    },
    "핵심 요지 파악": {
        "zero": "'핵심 요지 파악'이 0점입니다. 글 전체를 읽고 주제를 고르는 영역은 일부 표현만으로 답을 찾기 어려워 꾸준한 훈련이 필요합니다.",
        "perfect": "'핵심 요지 파악'이 만점입니다. 글 전체를 읽고 주제를 고르는 영역에서 어려운 지문의 내용을 잘 이해한 결과입니다.",
        "weak": "'핵심 요지 파악'의 득점률은 {rate}%입니다. 글 전체의 내용을 종합해 주제를 고르는 연습이 필요합니다.",
        "strong": "'핵심 요지 파악'의 득점률은 {rate}%입니다. 글 전체를 종합해 주제를 찾는 영역에서 어려운 지문의 내용을 잘 이해한 결과입니다.",
        "general": "'핵심 요지 파악'은 글 전체의 내용을 종합해 주제를 고르는 연습이 필요한 영역입니다.",
    },
    "내용 파악": {
        "zero": "'내용 파악'이 0점입니다. 지문과 선택지를 세세하게 대조하는 훈련이 필요합니다.",
        "perfect": "'내용 파악'이 만점입니다. 지문과 선택지를 세세하게 대조하며 어려운 지문의 내용을 빠르고 정확하게 이해한 결과입니다.",
        "weak": "'내용 파악'의 득점률은 {rate}%입니다. 지문과 선택지를 세세하게 대조하는 훈련이 필요합니다.",
        "strong": "'내용 파악'의 득점률은 {rate}%입니다. 지문과 선택지를 세세하게 대조하며 어려운 지문의 내용을 빠르고 정확하게 이해한 결과입니다.",
        "general": "'내용 파악'은 지문과 선택지를 세세하게 대조하는 훈련이 필요한 영역입니다.",
    },
    "문맥 파악": {
        "zero": "'문맥 파악'이 0점입니다. 앞뒤 문장의 내용을 함께 살펴 의미를 판단하는 꾸준한 훈련이 필요합니다.",
        "perfect": "'문맥 파악'이 만점입니다. 앞뒤 문장의 내용을 정확하게 연결해 어려운 지문의 내용을 잘 이해한 결과입니다.",
        "weak": "'문맥 파악'의 득점률은 {rate}%입니다. 앞뒤 문장의 내용을 함께 살펴 의미를 판단하는 꾸준한 훈련이 필요합니다.",
        "strong": "'문맥 파악'의 득점률은 {rate}%입니다. 앞뒤 문장의 내용을 정확하게 연결해 어려운 지문의 내용을 잘 이해한 결과입니다.",
        "general": "'문맥 파악'에서 실점이 확인되었습니다.",
    },
    # 규정서의 문구가 비어 있어 수치로 확인되는 사실만 안내합니다.
    "구조 파악": {
        "zero": "'구조 파악'이 0점입니다.",
        "perfect": "'구조 파악'이 만점입니다.",
        "weak": "'구조 파악'의 득점률은 {rate}%입니다.",
        "strong": "'구조 파악'의 득점률은 {rate}%입니다.",
        "general": "'구조 파악'에서 실점이 확인되었습니다.",
    },
    "추론": {
        "zero": "'추론'이 0점입니다.",
        "perfect": "'추론'이 만점입니다.",
        "weak": "'추론'의 득점률은 {rate}%입니다.",
        "strong": "'추론'의 득점률은 {rate}%입니다.",
        "general": "'추론'에서 실점이 확인되었습니다.",
    },
}


def render_area(item: dict[str, Any], state: str | None = None) -> str:
    selected_state = state or item["state"]
    template = AREA_MESSAGES[item["name"]][selected_state]
    return template.format(
        score=format_number(item["score"]),
        maximum=format_number(item["max"]),
        rate=format_derived(item["rate"]),
    )


def select_with_boundary_ties(items: list[dict[str, Any]], limit: int, reverse: bool) -> list[dict[str, Any]]:
    sorted_items = sorted(items, key=lambda x: ((-x["rate"] if reverse else x["rate"]), x["index"]))
    if len(sorted_items) <= limit:
        return sorted_items
    boundary = sorted_items[limit - 1]["rate"]
    if reverse:
        return [item for item in sorted_items if item["rate"] >= boundary]
    return [item for item in sorted_items if item["rate"] <= boundary]


def area_blocks(areas: list[dict[str, Any]]) -> tuple[list[str], list[str], list[str]]:
    strength_candidates = [item for item in areas if item["state"] in {"perfect", "strong"}]
    weakness_candidates = [item for item in areas if item["state"] in {"zero", "weak"}]
    strengths = [render_area(item) for item in select_with_boundary_ties(strength_candidates, 3, True)]

    if len(weakness_candidates) >= 4:
        sorted_weak = sorted(weakness_candidates, key=lambda x: (x["rate"], x["index"]))
        error_rates = [Decimal("100") - item["rate"] for item in weakness_candidates]
        low = min(error_rates)
        high = max(error_rates)
        broad = (
            "영역별로 보면 여러 영역에서 오답이 확인되었습니다. "
            f"영역별 오답률은 {format_derived(low)}%부터 {format_derived(high)}%까지 분포하여 "
            "전체 영역을 꾸준히 공부할 필요가 있습니다."
        )
        weaknesses = [broad, *[render_area(item, "general") for item in sorted_weak[:3]]]
    else:
        weaknesses = [render_area(item) for item in select_with_boundary_ties(weakness_candidates, 3, False)]
    # 규정표에서 일반 상태를 '출력'으로 정한 영역만 안내합니다.
    general_output_areas = {"핵심 요지 파악", "내용 파악"}
    generals = [
        render_area(item, "general")
        for item in sorted(areas, key=lambda x: x["index"])
        if item["state"] == "general" and item["name"] in general_output_areas
    ]
    return strengths, weaknesses, generals


def score_change_sentence(current: Decimal, previous: Decimal | None) -> str:
    if previous is None:
        return ""
    difference = current - previous
    if difference == 0:
        return "이전 평가와 같은 점수를 받았습니다."
    magnitude = format_number(abs(difference))
    if difference > 0:
        return f"이전 평가보다 {magnitude}점 향상되었습니다."
    return f"이전 평가보다 {magnitude}점 낮아졌습니다."


def class_average(analyses: dict[str, dict[str, Any]]) -> Decimal | None:
    scores = [analysis["current"] for analysis in analyses.values() if not analysis["absent"]]
    return sum(scores, Decimal("0")) / Decimal(len(scores)) if scores else None


def class_top_ids(analyses: dict[str, dict[str, Any]]) -> set[str]:
    scores = {sid: analysis["current"] for sid, analysis in analyses.items() if not analysis["absent"]}
    if not scores:
        return set()
    top_score = max(scores.values())
    return {sid for sid, score in scores.items() if score == top_score}


def remove_teacher_name(class_name: str) -> str:
    return re.sub(r"-[^-]+T$", "", class_name).strip("-")


def sanitize_windows_filename(name: str) -> str:
    return re.sub(r'[<>:"/\\|?*]', "_", name).rstrip(". ")


def combine_blocks(blocks: list[str]) -> str:
    cleaned = [block.strip() for block in blocks if clean_text(block)]
    return "\n\n".join(cleaned)


## 6. 전체 실행 함수

모든 반의 메시지를 메모리에서 먼저 만들고 검증한 뒤 저장합니다. 오류가 나면 어느 파일·학생·열에서 문제가 생겼는지 표시하고 최종 TXT는 저장하지 않습니다.


In [ ]:
def build_class_messages(
    data: ClassData,
    feedback: FeedbackInput,
    common_lesson: str,
    class_lesson: str,
    warnings: list[str],
) -> tuple[list[tuple[str, str, str]], dict[str, int]]:
    analyses = {row["학번"]: analyze_row(data, row) for row in data.rows}
    average = class_average(analyses)
    top_ids = class_top_ids(analyses)

    supplied_average = feedback.supplied_class_averages.get(data.class_name)
    if average is not None and supplied_average is not None:
        if abs(average - supplied_average) > Decimal("0.01"):
            warnings.append(
                f"FeedbackInput 학급평균과 정제 파일 계산값이 달라 정제 파일 계산값을 사용합니다: "
                f"학급명={data.class_name}, 입력={format_number(supplied_average)}, 계산={format_derived(average)}"
            )

    messages = []
    absent_count = 0
    previous_missing_count = 0
    for row in sorted(data.rows, key=lambda r: (r["학생명"], r["학번"])):
        sid = row["학번"]
        name = row["학생명"]
        analysis = analyses[sid]
        exclusions = feedback.student_exclusions.get(sid, set())

        if analysis["absent"]:
            absent_count += 1
            message = f"{friendly_subject(name)} ★★★ 시험 미실시 또는 중도 퇴원 ★★★"
            messages.append((name, sid, message))
            continue

        blocks = []
        if "CLASS_NOTE" not in exclusions:
            blocks.extend(feedback.class_blocks.get(data.class_name, []))
        if INCLUDE_CLASS_AVERAGE and average is not None and "CLASS_AVERAGE" not in exclusions:
            blocks.append(f"우리 반 평균은 {format_derived(average)}점입니다.")

        blocks.append(f"안녕하세요. {friendly_genitive(name)} {data.exam_name} 결과를 안내드립니다.")

        if "LESSON_CONTENT" not in exclusions:
            blocks.extend([text for text in (common_lesson, class_lesson) if text])
            blocks.extend(feedback.class_lessons.get(data.class_name, []))
            blocks.extend(feedback.student_lessons.get(sid, []))

        if "CURRENT_SCORE" not in exclusions:
            blocks.append(
                f"{friendly_subject(name)} 이번 시험에서 {format_number(analysis['current'])}점을 받았습니다."
            )

        if data.exam_no in {2, 4, 5} and "SCORE_CHANGE" not in exclusions:
            change = score_change_sentence(analysis["current"], analysis["previous"])
            if change:
                blocks.append(change)
            else:
                previous_missing_count += 1

        if INCLUDE_CLASS_FIRST_PLACE and sid in top_ids and "CLASS_RANK" not in exclusions:
            blocks.append("우리 반 1등입니다." if len(top_ids) == 1 else "우리 반 공동 1등입니다.")

        strengths, weaknesses, generals = area_blocks(analysis["areas"])
        if "AREA_STRENGTH" not in exclusions:
            blocks.extend(strengths)
        if "AREA_WEAKNESS" not in exclusions:
            blocks.extend(weaknesses)
        if "AREA_GENERAL" not in exclusions:
            blocks.extend(generals)

        if "STUDENT_NOTE" not in exclusions:
            blocks.extend(feedback.student_blocks.get(sid, []))

        closing = clean_text(CLOSING_BY_EXAM.get(data.exam_no, ""))
        if closing:
            blocks.append(normalize_manual_text(closing))

        message = combine_blocks(blocks)
        if len(message) > MAX_RECOMMENDED_MESSAGE_CHARS:
            warnings.append(
                f"권장 글자 수 초과: 파일명={data.path.name}, 학번={sid}, "
                f"글자수={len(message)}, 권장={MAX_RECOMMENDED_MESSAGE_CHARS}"
            )
        messages.append((name, sid, message))

    return messages, {
        "students": len(messages),
        "absent": absent_count,
        "previous_missing": previous_missing_count,
    }


def choose_output_path(output_dir: Path, year: int, exam_no: int, class_name: str) -> Path:
    display_class = remove_teacher_name(class_name)
    base_name = sanitize_windows_filename(f"[{year}-{exam_no}차] {display_class}-피드백.txt")
    target = output_dir / base_name
    if target.exists():
        stamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
        target = output_dir / f"{target.stem}_{stamp}{target.suffix}"
    return target


def render_class_txt(messages: list[tuple[str, str, str]]) -> str:
    sections = [f"▷ {name}\n\n{message}" for name, _, message in messages]
    return "\n\n--------------------------\n\n".join(sections) + "\n"


def run_feedback_generation(
    data_clean_path: Path = DATA_CLEAN_PATH,
    feedback_input_path: Path = FEEDBACK_INPUT_PATH,
    output_path: Path = FEEDBACK_OUTPUT_PATH,
) -> dict[str, Any]:
    data_clean_path = Path(data_clean_path)
    feedback_input_path = Path(feedback_input_path)
    output_path = Path(output_path)
    if not data_clean_path.exists():
        raise FeedbackError(f"DataClean 폴더를 찾을 수 없습니다: {data_clean_path}")

    files = sorted(
        path for path in data_clean_path.glob("*-정제.xlsx")
        if not path.name.startswith("~$")
    )
    if not files:
        raise FeedbackError(f"DataClean 폴더에 '*-정제.xlsx' 파일이 없습니다: {data_clean_path}")

    warnings = []
    classes = []
    for path in files:
        loaded, file_warnings = load_class_file(path)
        classes.append(loaded)
        warnings.extend(file_warnings)

    metadata = {(item.year, item.exam_no) for item in classes}
    if len(metadata) != 1:
        details = sorted((item.path.name, item.year, item.exam_no) for item in classes)
        raise FeedbackError(
            "[오류] 연도와 차수가 다른 시험 정보는 Legacy 폴더로 옮겨 주세요. "
            f"감지 결과={details}"
        )
    year, exam_no = next(iter(metadata))

    all_ids = [row["학번"] for item in classes for row in item.rows]
    duplicate_ids = sorted({sid for sid in all_ids if all_ids.count(sid) > 1})
    if duplicate_ids:
        locations = {
            sid: [item.path.name for item in classes if any(row["학번"] == sid for row in item.rows)]
            for sid in duplicate_ids
        }
        raise FeedbackError(f"동일 차수의 반별 파일에 학번이 중복되었습니다: {locations}")

    students_by_id = {row["학번"]: row for item in classes for row in item.rows}
    class_names = {item.class_name for item in classes}
    feedback = load_feedback_input(feedback_input_path, year, exam_no, students_by_id, class_names)
    warnings.extend(feedback.warnings)

    common_lesson = read_text_file(COMMON_LESSON_TEXT_PATH)
    class_lessons = {
        class_name: read_text_file(path)
        for class_name, path in CLASS_LESSON_TEXT_PATHS.items()
    }
    unknown_lesson_classes = set(class_lessons) - class_names
    if unknown_lesson_classes:
        raise FeedbackError(f"정제 파일에 없는 학급의 수업내용 경로가 설정되었습니다: {sorted(unknown_lesson_classes)}")

    # 모든 메시지와 파일명을 먼저 확정합니다. 이 아래 검증이 끝나기 전에는 저장하지 않습니다.
    prepared = []
    summary = {"students": 0, "absent": 0, "previous_missing": 0}
    output_names = set()
    for item in classes:
        messages, counts = build_class_messages(
            item,
            feedback,
            common_lesson,
            class_lessons.get(item.class_name, ""),
            warnings,
        )
        target = choose_output_path(output_path, year, exam_no, item.class_name)
        if target.name in output_names:
            raise FeedbackError(f"서로 다른 학급의 출력 파일명이 충돌합니다: {target.name}")
        output_names.add(target.name)
        prepared.append((target, render_class_txt(messages)))
        for key in summary:
            summary[key] += counts[key]

    output_path.mkdir(parents=True, exist_ok=True)
    saved = []
    for target, content in prepared:
        temporary = target.with_suffix(target.suffix + ".tmp")
        temporary.write_text(content, encoding="utf-8-sig", newline="\n")
        temporary.replace(target)
        saved.append(target)

    print(f"시험연도: {year}")
    print(f"시험차수: 제{exam_no}차")
    print(f"처리한 반 수: {len(classes)}")
    print(f"처리한 학생 수: {summary['students']}")
    print(f"결시 학생 수: {summary['absent']}")
    print(f"이전점수 결측 학생 수: {summary['previous_missing']}")
    print(f"경고 수: {len(warnings)}")
    print("저장한 TXT:")
    for path in saved:
        print(f"- {path}")
    if warnings:
        print("\n[경고]")
        for warning in warnings:
            print(f"- {warning}")

    return {
        "year": year,
        "exam_no": exam_no,
        "class_count": len(classes),
        **summary,
        "warnings": warnings,
        "saved": saved,
    }


## 7. 내장 테스트

실제 파일을 쓰기 전에 이름 조사, 점수 형식, 점수 변화, 영역 분석, 결시 문구를 작은 가상 자료로 검사합니다.


In [ ]:
def run_self_tests() -> None:
    assert friendly_subject("김주원") == "주원이는"
    assert friendly_subject("이지우") == "지우는"
    assert friendly_genitive("김주원") == "주원이의"
    assert friendly_genitive("이지우") == "지우의"
    assert format_number(Decimal("48.0")) == "48"
    assert format_number(Decimal("48.5")) == "48.5"
    assert score_change_sentence(Decimal("80"), Decimal("70")) == "이전 평가보다 10점 향상되었습니다."
    assert score_change_sentence(Decimal("70"), Decimal("80")) == "이전 평가보다 10점 낮아졌습니다."
    assert score_change_sentence(Decimal("80"), Decimal("80")) == "이전 평가와 같은 점수를 받았습니다."

    sample = ClassData(
        path=Path("[2026-2차] 테스트반-정제.xlsx"),
        year=2026,
        exam_no=2,
        exam_name="2026년 제2차 정기평가",
        class_name="테스트반",
        rows=[{
            "학급명": "테스트반", "학생명": "김주원", "학번": "00781", "점수": "80",
            "어휘 파악": "17", "핵심 요지 파악": "7", "내용 파악": "18",
            "문맥 파악": "10", "구조 파악": "10", "추론": "18", "이전점수": "70",
            "_excel_row": 4,
        }],
    )
    analysis = analyze_row(sample, sample.rows[0])
    assert analysis["current"] == Decimal("80")
    assert sum((area["score"] for area in analysis["areas"]), Decimal("0")) == Decimal("80")
    strengths, weaknesses, generals = area_blocks(analysis["areas"])
    assert strengths
    assert isinstance(weaknesses, list)
    assert isinstance(generals, list)

    absent = ClassData(
        path=Path("[2026-1차] 테스트반-정제.xlsx"),
        year=2026,
        exam_no=1,
        exam_name="2026년 제1차 정기평가",
        class_name="테스트반",
        rows=[{
            "학급명": "테스트반", "학생명": "박민준", "학번": "00001", "점수": "",
            **{area: "" for area in AREA_ORDER}, "_excel_row": 4,
        }],
    )
    empty_feedback = FeedbackInput({}, {}, {}, {}, {}, {}, [])
    messages, counts = build_class_messages(absent, empty_feedback, "", "", [])
    assert counts["absent"] == 1
    assert messages[0][2] == "민준이는 ★★★ 시험 미실시 또는 중도 퇴원 ★★★"
    print("내장 테스트 통과: 2-1 핵심 규칙이 정상 작동합니다.")


run_self_tests()


## 8. 실제 TXT 생성

위 설정 경로가 맞는지 확인한 뒤 이 셀을 실행하세요. 반환값의 `saved`에 생성된 파일 경로가 표시됩니다.


In [ ]:
result = run_feedback_generation()
result


## 현재 2-1에서 의도적으로 보류한 부분

- 제3·6차 누적평가, 주간테스트·과제수행 점수 평가, 진급/반 유지: 2-2에서 구현
- 시험 차수별 마무리 문구: 규정 확정 후 `CLOSING_BY_EXAM`에 입력
- `구조 파악`·`추론`의 상세 해석 문구: 규정서 문구가 확정되기 전까지 중립적인 득점 사실만 출력
- 수업 태도·과제 원인 추정: 하지 않음. `FeedbackInput.xlsx`의 실제 관찰 문구만 삽입
